# Ad Bidding Optimization - Exploratory Data Analysis

In this notebook, we explore the synthetic ad auction dataset generated in Phase 2. We will analyze key business metrics such as CTR, ROI, and Cost, as well as investigate traffic quality signals (fraud).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add the project root to python path to access local src modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.config.settings import settings

import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Load Data Quality Checks

In [ ]:
# Load Preprocessed Data
processed_path = os.path.join(settings.PROCESSED_DATA_PATH, "ad_auction_processed.csv")
df = pd.read_csv(processed_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Dataset Shape: {df.shape}")

# Null Value Check
null_counts = df.isnull().sum()
print("\n--- Null values per column ---")
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No null values found!")

# Logic Constraints Check
invalid_bids = df[df['bid_price'] < df['floor_price']]
print(f"\nAuctions where Bid < Floor (Logic errors): {len(invalid_bids)}")

negative_costs = df[df['cost'] < 0]
print(f"Auctions resulting in negative cost: {len(negative_costs)}")


## 2. Feature Profiling: CTR and Conversion Rates

In [ ]:
# Global Metrics
global_ctr = df['actual_click'].mean()
global_cvr = df[df['actual_click'] == 1]['conversion'].mean() # CVR given click
print(f"Global CTR: {global_ctr:.2%}")
print(f"Global Conv Rate (post-click): {global_cvr:.2%}")

print("\n--- Top 5 Publishers by CTR ---")
# Quality by Publisher
pub_stats = df.groupby('publisher_id').agg(
    impressions=('request_id', 'count'),
    clicks=('actual_click', 'sum'),
    conversions=('conversion', 'sum'),
    avg_roi=('roi', 'mean'),
).reset_index()

pub_stats['ctr'] = pub_stats['clicks'] / pub_stats['impressions']
pub_stats.sort_values('ctr', ascending=False).head(5)

## 3. Spend vs Bid Distribution Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['bid_price'], bins=50, color='blue', alpha=0.5, label='Bid Price', kde=True)
sns.histplot(df['cost'], bins=50, color='orange', alpha=0.5, label='Actual Cost (Impression Pay)', kde=True)
plt.title('Bid Price vs Clearing Cost Distribution')
plt.xlabel('Price (USD)')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Categorical Impact on Clicks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=df, x='device_type', y='actual_click', ax=axes[0])
axes[0].set_title('CTR by Device Type')
axes[0].set_ylabel('CTR')

sns.barplot(data=df, x='exchange_id', y='actual_click', ax=axes[1])
axes[1].set_title('CTR by Exchange')
axes[1].set_ylabel('CTR')

plt.tight_layout()
plt.show()